---
title: Haplotype Phasing Results
date: "9999-12-31"
edit_url: null
---

In [5]:
blockfile = "~/blocks.summary.gz"
blockfile = "/home/pdimens/Documents/harpy/Phase/snp/reports/blocks.summary.gz"
contigs = "default"

In [6]:
import altair as alt
from harpy.report.components import colored_boxes, print_html, standard_itable
from harpy.report.utilities import binned_histogram, nxx
from harpy.report.theme import get_okabe, palette
import itables
import os
import numpy as np
import pandas as pd
import sys
itables.init_notebook_mode(connected=True)

In [65]:
df = pd.read_table(blockfile, sep = '\t')
if len(df) == 0:
    print_html(f"No phase blocks were observed in the input file {os.path.basename(blockfile)}")
    sys.exit(0)

df['pos_end'] = df['pos_start'] + df['block_length']
df = df[df['block_length'] > 0].iloc[:, [0,1,2,3,5,4]]

The data presented here are the results of phasing genotypes into haplotypes using
[HapCut2](https://github.com/vibansal/HapCUT2). The information is derived from `data/reports/blocks.summary.gz`,
which summarized information across all samples using the `.blocks` files HapCut2
generates[^1]. This page shows general and per-contig information. The `Per-Contig Metrics` section will show you information when aggregating across samples within a contig, whereas the `Per-Sample Metrics` section will show you information relating to haplotypes in individual samples. Haplotype blocks with a size of `0` are ignored in the reporting below.

[^1]:
    The VCF files HapCut2 also generates were not used as they lack
    some of the information in the `.blocks` files that were collated in this report.

In [8]:
overall = {
    'Total SNPs': df['n_snp'].sum(),
    'Mean SNPs': round(df['n_snp'].mean(), 0),
    'Median SNPs': df['n_snp'].median(),
    'Longest Haplotype': df['block_length'].max(),
    'N50': nxx(df['block_length'], 50),
    'N75': nxx(df['block_length'], 75),
    'N90': nxx(df['block_length'], 90)
}

boxes = colored_boxes()
for k,v in overall.items():
    boxes.add(v, k)
boxes.render()

## Overall Metrics
### Haplotype Lengths
This is the distribution of haplotype length (in base pairs). 
Phasing typically has a few very large haplotypes, resulting in extreme right-tails in these distributions.

In [74]:
_hist = binned_histogram(df['block_length']/1000, 1, normalize = True)

_chart = (
    alt.Chart(_hist)    
    .transform_calculate(
        interval_kbp ='datum.interval + " kbp"'
        )
    .mark_area()
    .encode(
        x=alt.X('bin:Q', title="Haplotype Length (kbp)").axis(tickMinStep = 5,labelAngle=-40),
        y=alt.Y('proportion:Q', title='Percent of Haplotypes').scale(domain = [0,1]).axis(format='%'),
        tooltip = [
            alt.Tooltip("interval_kbp:N", title="Length"),
            alt.Tooltip("proportion:Q", format = '.2%', title = "% Haplotypes")
        ]
    )
).interactive()

nxx_table = pd.DataFrame({'stat': ['N50', 'N75', 'N90'], 'value': [overall['N50']/1000, overall['N75']/1000, overall['N90']/1000]})

nxx_lines = (
    alt.Chart(nxx_table)
    .mark_rule(size = 2, strokeDash = (4,4))
    .encode(
        x='value:Q',
        color = alt.Color('stat:N').scale(range = get_okabe([0,2,6])),
        tooltip = ['stat', 'value']
    )
)

(
    (_chart + nxx_lines)
    .configure_legend(orient = 'top')
    .properties(
        title = "Haplotype Block Lengths",
        usermeta={'embedOptions': {'downloadFileName': 'haplotype.length'}}
    )
)

alt.LayerChart(...)

### NX Information

An **NX** metric (e.g. **N50**) is the length of the shortest molecule in the group of longest molecules that together
represent at least **X%** of the total molecules by length. For example, `N50` would be the shortest molecule in the 
group of longest molecules that together represent **50%** of the total molecules by length (sort of like a cumulative median).

In [72]:
nxx_df = df.groupby(['sample', 'contig']).agg(
    N50=('block_length', lambda x: nxx(x, 50)/1000),
    N75=('block_length', lambda x: nxx(x, 75)/1000),
    N90=('block_length', lambda x: nxx(x, 90)/1000)
).reset_index()
nxx_df

(
    alt.Chart(nxx_df)
    .transform_fold(['N50', 'N75', 'N90'], as_ = ['Metric', 'NXX'])
    .transform_density('NXX', resolve = 'independent', groupby = ['Metric'])
    .mark_area()
    .encode(
        x = alt.X('value:Q', title = 'Length (kbp)', bin = 'binned'),
        y = alt.Y('density:Q', title = "Density").stack(False).axis(labels = False),
        color = alt.Color('Metric:N').legend(None),
        row = alt.Row('Metric:N', title = None)
    )
    .resolve_scale(x='independent', y='independent')
    .configure_legend(orient = 'top')
    .properties(
        height = 175, width = 650,
        title = alt.Title("NX Stats Across Samples and Contigs", subtitle = 'Distributions derived from NX per sample per contig'),
        usermeta={'embedOptions': {'downloadFileName': 'haplotype.length'}}
    )
)

alt.Chart(...)

## Per-Contig Metrics

In [73]:
percontig = df.groupby('contig').agg(
    mean_snps=('n_snp', lambda x: round(x.mean(), 0)),
    median_snps=('n_snp', 'median'),
    mean_blocksize=('block_length', lambda x: round(x.mean(), 0)),
    max_blocksize=('block_length', 'max'),
    N50=('block_length', lambda x: nxx(x, 50)),
    N75=('block_length', lambda x: nxx(x, 75)),
    N90=('block_length', lambda x: nxx(x, 90))
).reset_index()
percontig.columns = ['Contig', 'Mean SNPs/Hap', 'Median SNPs/Hap', 'Mean Length', 'Max Length', 'N50', 'N75', 'N90']
standard_itable(percontig, filename = 'haplotypes.contigs', fixedcols = 1)

Loading ITables v2.6.2 from the internet... (need help?)


In [ ]:
dropdown_buttons <- function(CONTIGS, CUTOFF = 0){
  buttonlist <- list()
  idx <- 0
  n_contigs <- length(CONTIGS)
  for(i in CONTIGS){
    idx <- idx + 1
    if((CUTOFF > 0) && (idx > CUTOFF)) break
    visibility <- as.list(rep(FALSE, n_contigs))
    visibility[[idx]] <- TRUE
    buttonlist[[idx]] <- list(method = "restyle", label = i, args = list("visible", visibility))
  }
  return( list(list(y = 1, buttons = buttonlist)) )
}

In [ ]:
plotting_contigs <- contigs
if (all(plotting_contigs == "default")){
  .contigs <- group_by(df, contig) %>%
    summarize(size = max(pos_end)) %>%
    arrange(desc(size))
  plot_contigs <- .contigs$contig[1:min(nrow(.contigs), 40)] 
  } else {
  plot_contigs <- plotting_contigs
}

### Haplotype Lengths Per Contig
This is the distribution of haplotype length (in base pairs) for 
each contig, up to 300 contigs. Phasing typically has a few very large haplotypes, resulting in extreme 
right-tails in these distributions, therefore, the haplotype lengths are **log-scaled lengths**
to collapse the right tail for better readability. The dotted vertical bar represents the mean haplotype length.

In [ ]:
fig <- plot_ly(hoverinfo = "none") %>%
  layout(
    xaxis = list(title = "Log-Scaled Haplotype Length (bp)", fixedrange = TRUE),#, type = "log"),
    yaxis = list(fixedrange = TRUE),
    title = "Haplotype Lengths by Contig",
    showlegend = F,
    updatemenus = dropdown_buttons(plot_contigs)
  ) %>% config(displayModeBar = FALSE)
# Loop over the categories
for (cont in plot_contigs) {
  subset_cont <- df$block_length[df$contig == cont]
  fig <- fig %>%
    add_trace(
      x = log(subset_cont),
      type = "violin",
      name = cont,
      side = "positive",
      meanline = list(visible = T),
      visible = "legendonly"
    )
}

fig

### Haplotype Stats by Contig

In [ ]:
column_description <- c(
  "name of the contig",
  "total haplotypes on the contig",
  "mean number of SNPs per haplotype",
  "median number of SNPs per haplotype",
  "mean length of haplotypes, given in base pairs",
  "N50 length of the haplotype blocks, given in base pairs",
  "length of the largest haplotype, given in base pairs"
  )

headerCallback <- c(
  "function(thead, data, start, end, display){",
  paste0(" var tooltips = ['", paste(column_description, collapse = "','"), "'];"),
  "  for(var i=0; i<tooltips.length; i++){",
  "    $('th:eq('+i+')',thead).attr('title', tooltips[i]);",
  "  }",
  "}"
)

DT::datatable(
  percontig,
  rownames = F,
  extensions = 'Buttons',
  colnames = c("Contig", "Total Haplotypes", "Mean SNPs", "Median SNPs", "Mean Haplotype Length", "Haplotype N50", "Largest Haplotype"),
  fillContainer = T,
  options = list(
    dom = 'Brtp',
    buttons = list(list(extend = "csv",filename = "phasing_per_contig")), 
    scrollX = TRUE,
    headerCallback = JS(headerCallback)
  )
)

## Per-Sample Metrics
The plots below shows the distribution of haplotype lengths (in base pairs) for each 
sample. While no Y-axis for these counts are provided, this plot is intended to be more of
a visual reference of the relative distributions of haplotype lengths among samples.
Phasing typically has a few very large haplotypes, resulting in extreme 
right-tails in these distributions, therefore, the haplotype lengths are
**log-scaled lengths** to collapse the right tail for better readability.
The dotted vertical bar represents the mean haplotype length. 


In [ ]:
persample <- df %>% group_by(sample) %>% summarise(
    n_haplo = sum(length(n_snp)),
    mean_snps = round(mean(n_snp), digits = 0),
    median_snps = median(n_snp),
    mean_blocksize = round(mean(block_length), digits = 0),
    N50 = NX(block_length, 50),
    max_blocksize = max(block_length)
)

column_description <- c(
  "name of the sample",
  "number of haplotypes",
  "mean number of SNPs per haplotype",
  "median number of SNPs per haplotype",
  "mean length of haplotypes, given in base pairs",
  "N50 length of the haplotype blocks, given in base pairs",
  "length of the largest haplotype, given in base pairs"
)

headerCallback <- c(
  "function(thead, data, start, end, display){",
  paste0(" var tooltips = ['", paste(column_description, collapse = "','"), "'];"),
  "  for(var i=0; i<tooltips.length; i++){",
  "    $('th:eq('+i+')',thead).attr('title', tooltips[i]);",
  "  }",
  "}"
)

DT::datatable(
  persample,
  rownames = F,
  extensions = 'Buttons',
  colnames = c("Sample", "Haplotypes", "Mean SNPs", "Median SNPs", "Mean Haplotype Length", "Haplotype N50", "Largest Haplotype"),
  fillContainer = T,
  options = list(
    dom = 'Brtp', 
    buttons = list(list(extend = "csv",filename = "phasing_per_sample")), 
    scrollX = TRUE,
    headerCallback = JS(headerCallback)
  )
)

### the ridgeplot
CONSIDER REMOVING THIS

In [ ]:
fig <- plot_ly(hoverinfo = "none") %>%
  layout(
    xaxis = list(title = "Log-Scaled Haplotype Length (bp)", fixedrange = TRUE),
    yaxis = list(fixedrange = TRUE),
    title = "Haplotype Lengths by Sample",
    showlegend = F,
    updatemenus = dropdown_buttons(levels(df$sample), 600)
  ) %>% config(displayModeBar = FALSE)
# Loop over the categories
.idx <- 0
for (samp in levels(df$sample)) {
  .idx <- .idx + 1
  if(.idx > 600) break
  subset_sample <- df$block_length[df$sample == samp] 
  fig <- fig %>%
    add_trace(
      x = log(subset_sample),
      type = "violin",
      name = samp,
      side = "positive",
      meanline = list(visible = T),
      visible = "legendonly"
    )
}

fig